# Attention Head Taxonomy

Systematic classification of attention heads by functional role: induction,
previous-token, copy, inhibition, and unclassified.

References:
- Olsson et al. (2022) "In-context Learning and Induction Heads"
- Wang et al. (2023) "Interpretability in the Wild"

## Why This Matters

Attention head taxonomy reveals how heads route information between positions. Understanding attention mechanics is central to reverse-engineering transformer circuits, as attention patterns determine what information each component can access and transform.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [Olsson et al. (2022) "In-context Learning and Induction Heads"](https://arxiv.org/abs/2209.11895) — Discovers induction heads and training phase transitions

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.attention_head_taxonomy import (
    induction_head_score,
    previous_token_head_score,
    copy_head_score,
    inhibition_head_score,
    head_taxonomy_summary,
)

In [ ]:
cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
# Tokens with repetitions (needed for induction head detection)
tokens = jnp.array([0, 5, 10, 15, 0, 5, 10, 15, 20])
metric_fn = lambda logits: float(logits[-1, 0] - logits[-1, 1])
print("Model ready")

## Induction Head Scores

Induction heads attend to the token after a previous occurrence of the current token.
They are key to in-context learning.

In [ ]:
result = induction_head_score(model, tokens)
print(f"Induction scores shape: {result['scores'].shape}")
print(f"Max score: {result['max_score']:.4f}")
print(f"\nTop induction heads:")
for l, h, s in result['top_heads']:
    print(f"  L{l}H{h}: {s:.4f}")

## Previous-Token Head Scores

Previous-token heads attend primarily to position q-1.

In [ ]:
result = previous_token_head_score(model, tokens)
print(f"Max score: {result['max_score']:.4f}")
print(f"\nTop previous-token heads:")
for l, h, s in result['top_heads'][:5]:
    print(f"  L{l}H{h}: {s:.4f}")

## Copy Head Scores

Copy heads promote attended-to tokens into the output prediction.

In [ ]:
result = copy_head_score(model, tokens)
print(f"\nTop copy heads:")
for l, h, s in result['top_heads'][:5]:
    print(f"  L{l}H{h}: {s:.4f}")
print(f"\nCopied tokens:")
for (l, h), toks in list(result['copied_tokens'].items())[:5]:
    print(f"  L{l}H{h}: {toks}")

## Inhibition Head Scores

Inhibition heads reduce the metric when present.

In [ ]:
result = inhibition_head_score(model, tokens, metric_fn)
print(f"Number of inhibitory heads: {result['n_inhibitory']}")
print(f"\nTop inhibition heads:")
for l, h, s in result['top_heads'][:5]:
    print(f"  L{l}H{h}: {s:.4f}")

## Full Taxonomy Summary

Classify every head by its dominant functional role.

In [ ]:
result = head_taxonomy_summary(model, tokens, metric_fn)
print("Head classifications:")
for (l, h), t in sorted(result['classifications'].items()):
    print(f"  L{l}H{h}: {t}")
print(f"\nType counts: {result['type_counts']}")
print(f"Type distribution: {result['type_distribution']}")